## 0. Paths and Config

In [ ]:
# Set SNT Path
ROOT_PATH <- "~/workspace"

In [ ]:
# Pipeline path
PIPELINE_PATH <- file.path(ROOT_PATH, "pipelines", "snt_user_population")

# Load shared utils, then pipeline utils (the pipeline utils expect snt_utils.r to be loaded)
source(file.path(ROOT_PATH, "code", "snt_utils.r"))
source(file.path(ROOT_PATH, "code", "snt_palettes.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_user_population_report.r"))
snt_paths <- init_snt_workspace(
    snt_pipeline_name = "snt_user_population",
    snt_root_path = ROOT_PATH,
    packages = c("tidyverse", "arrow", "sf", "reticulate", "glue", "classInt", "IRdisplay")
)

### 0.1 Load configuration

In [ ]:
# Load config
config_json <- load_snt_config(file.path(snt_paths$CONFIG_PATH, "SNT_config.json"))

# Configuration variables
DATASET_POP_USER_ID <- config_json$SNT_DATASET_IDENTIFIERS$SNT_POPULATION_USER_PROVIDED
DATASET_FORMATTING_ID <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

## 1. Import data

In [ ]:
# User population data from dataset
population_data <- load_dataset_file(DATASET_POP_USER_ID, paste0(COUNTRY_CODE, "_population.parquet"))

# Rows without a valid YEAR cannot be placed on a yearly map (reported by the pipeline)
n_no_year <- sum(is.na(population_data$YEAR))
if (n_no_year > 0) {
    log_msg(glue("{n_no_year} row(s) without YEAR are not displayed."), "warning")
    population_data <- population_data %>% dplyr::filter(!is.na(YEAR))
}
printdim(population_data)

In [ ]:
# Shapes data from formatting dataset
shapes_data <- load_dataset_file(DATASET_FORMATTING_ID, paste0(COUNTRY_CODE, "_shapes.geojson"))
printdim(shapes_data)

### 1.1 Check admin units against the shapes

In [ ]:
# Stops with [ERROR] if a unit of the file does not match the shapes (names or IDs)
check_population_matches_shapes(population_data, shapes_data, COUNTRY_CODE)

## 2. Carte des populations par niveau administratif 2

> Compute plot breaks

In [ ]:
# Indicator columns: everything except the metadata columns, skipping empty ones
metadata_columns <- c("YEAR", ADM_COLUMNS)
indicators <- setdiff(colnames(population_data), metadata_columns)
empty_indicators <- indicators[sapply(indicators, function(col) all(is.na(population_data[[col]])))]
if (length(empty_indicators) > 0) {
    log_msg(glue("Indicator(s) without values, not displayed: {paste(empty_indicators, collapse = ', ')}"))
}
indicators <- setdiff(indicators, empty_indicators)

n_classes <- 6  # one colour per class in palette_population

# Classes are computed over all years, so the maps of one indicator share a legend
set.seed(1)  # kmeans classification is random
label_breaks_list <- list()
for (indicator in indicators) {
    thresholds <- compute_value_intervals(population_data[[indicator]], n_classes, style = "kmeans")
    label_breaks_list[[indicator]] <- list(
        thresholds = thresholds,
        labels = create_dynamic_labels(thresholds)
    )
}

> Generate plots

In [ ]:
palette_population <- c(
    "1" = "#fae6db",
    "2" = "#f1b195",
    "3" = "#ea7354",
    "4" = "#cc3f32",
    "5" = "#972620",
    "6" = "#5e0f0f"
)

# One map per indicator, one facet per year (3 per row)
n_facet_rows <- ceiling(dplyr::n_distinct(population_data$YEAR) / 3)

for (indicator in indicators) {
    plot <- build_population_choropleth(
        population_data = population_data %>% dplyr::select(dplyr::all_of(c(metadata_columns, indicator))),
        shapes_data = shapes_data,
        population_column = indicator,
        thresholds = label_breaks_list[[indicator]]$thresholds,
        labels = label_breaks_list[[indicator]]$labels,
        legend_title = glue("{indicator} :"),
        plot_title = glue("{indicator} par niveau administratif 2"),
        palette_values = palette_population
    )

    # Saved as PNG and embedded as an image to keep the report light
    save_and_display_plot(
        plot = plot,
        output_file = file.path(snt_paths$OUTPUT_PLOTS_PATH, glue("{COUNTRY_CODE}_choropleth_population_user_{indicator}.png")),
        width = 21,
        height = 6 + 9 * n_facet_rows
    )
}